# SentiSense — Analysis, Modeling & Explainability

The post-ingestion **last stages** (features → clustering → baselines → LSTM tuning →
holdout → backtest) with **visualizations** and **explainability**. Reuses the
`sentisense` package — no logic is duplicated here; this notebook is the *viz/results*
surface over the same functions the CLI pipeline runs.

**Hard invariants** (enforced inside the package): cutoff `<= 2023-10-07`, train-only
scaling/PCA, chronological / TimeSeriesSplit validation, sacred held-out test.

## Prerequisites
```bash
uv sync --extra ml --extra embed --extra finance --extra notebook   # at repo root
# .env has the DB URL + local LLM backend
```
Heavy explainability deps (shap, umap, seaborn) are also lazy-installed in §0 if missing.

## Sections
0. Setup  · 1. Coverage & EDA  · 2. Features  · 3. Embeddings & clustering  ·
4. Baselines + SHAP/permutation explainability  · 5. LSTM tuning (Optuna viz)  ·
6. Holdout: ROC/PR, calibration, threshold, confusion  · 7. Backtest (equity / Sharpe / drawdown)  ·
8. Verdict


## 0. Setup

In [ ]:
# Lazy-install analysis/explainability extras if absent (heavy; not in base env).
import importlib, subprocess, sys
for pkg, imp in [("shap", "shap"), ("umap-learn", "umap"), ("seaborn", "seaborn")]:
    if importlib.util.find_spec(imp) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)


In [ ]:
from __future__ import annotations
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import text

import sentisense  # loads .env
from sentisense.db import get_engine, get_connection_url
from sentisense.constants import CUTOFF_DATE

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)
engine = get_engine()
print("DB:", get_connection_url().rsplit("@", 1)[-1])
print("Cutoff:", CUTOFF_DATE)


def feat_family(col: str) -> str:
    """Bucket a feature column into its signal source (for grouped explainability)."""
    if col.startswith(("mean_", "ix_")) or col == "n_headlines":
        return "news"
    if col.startswith(("SP500_", "VIX_", "Brent_", "USDILS_", "VTA35_", "Market_", "FX_")):
        return "cross_asset"
    if col.startswith(("TA125_", "DoW_")):
        return "technical"
    return "other"


## 1. Coverage & EDA
Corpus reach, per-model scored breakdown, monthly volume.

In [ ]:
models = pd.read_sql(text('''
    SELECT nv.model_name,
           COUNT(*) FILTER (WHERE nv.validation_passed) AS validated,
           COUNT(*) AS total
    FROM nlp_vectors nv JOIN raw_headlines rh ON rh.id = nv.headline_id AND rh.date <= :c
    GROUP BY nv.model_name ORDER BY validated DESC
'''), engine, params={"c": CUTOFF_DATE})
display(models)

monthly = pd.read_sql(text('''
    SELECT to_char(date_trunc('month', date), 'YYYY-MM') AS month, COUNT(*) AS raw
    FROM raw_headlines WHERE date <= :c GROUP BY 1 ORDER BY 1
'''), engine, params={"c": CUTOFF_DATE})
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(pd.to_datetime(monthly["month"]), monthly["raw"], lw=1.2)
ax.set_title("Headline volume per month (<= cutoff)"); ax.set_ylabel("headlines"); plt.show()


## 2. Features
Build the leak-safe daily frames. Inspect feature groups + correlations + target balance over time.

In [ ]:
from sentisense.features import build_datasets
mt, ml = build_datasets(engine)   # daily-mean (tree shape), per-source (LSTM shape)
print("mt (daily-mean):", mt.shape, "| ml (per-source):", ml.shape)
print("target +rate:", f"{mt['Target'].mean():.3f}")

fam_counts = pd.Series([feat_family(c) for c in mt.columns if c != "Target"]).value_counts()
print("feature families:", fam_counts.to_dict())


In [ ]:
# Target rate by year — is the up/down balance stable, or regime-drifting?
yr = mt.assign(year=mt.index.year).groupby("year")["Target"].agg(["mean", "size"])
fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(yr.index.astype(str), yr["mean"], color="#3b82f6"); ax.axhline(0.5, ls=":", c="gray")
ax.set_title("Next-day UP rate by year (class balance drift)"); ax.set_ylabel("up rate"); plt.show()

# Correlation of the news + interaction features with the target.
news_ix = [c for c in mt.columns if c.startswith(("mean_", "ix_"))]
corr = mt[news_ix + ["Target"]].corr()["Target"].drop("Target").sort_values()
fig, ax = plt.subplots(figsize=(8, max(3, 0.3 * len(corr))))
corr.plot.barh(ax=ax, color=np.where(corr > 0, "#10b981", "#ef4444"))
ax.set_title("News/interaction feature <-> Target correlation"); plt.show()


## 3. Embeddings & narrative clustering
2-D map of daily e5 centroids colored by next-day direction; dominant-cluster-ratio over time.

In [ ]:
from sentisense.embed import load_embeddings
meta, vecs = load_embeddings(engine)
print("cached embeddings:", len(meta), "dim:", (vecs.shape[1] if vecs.size else 0))
if vecs.size:
    # daily centroid -> UMAP 2-D, colored by that day's next-day direction
    cen = pd.DataFrame(vecs, index=pd.to_datetime(meta["date"].values)).groupby(level=0).mean()
    common = cen.index.intersection(mt.index)
    import umap
    emb2d = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(cen.loc[common].values)
    lab = mt.loc[common, "Target"].values
    fig, ax = plt.subplots(figsize=(7, 6))
    sc = ax.scatter(emb2d[:, 0], emb2d[:, 1], c=lab, cmap="coolwarm", s=6, alpha=0.6)
    ax.set_title("Daily news-embedding centroids (UMAP), colored by next-day TA-125 direction")
    plt.colorbar(sc, label="1=up"); plt.show()
else:
    print("No embeddings yet — run the embed stage first.")


In [ ]:
from sentisense.cluster import build_narrative_features
nf = build_narrative_features(engine)
if nf is not None and not nf.empty:
    fig, ax = plt.subplots(figsize=(14, 3))
    nf["dominant_cluster_ratio"].rolling(20).mean().plot(ax=ax, color="#7c3aed")
    ax.set_title("Dominant-cluster ratio (20-day MA) — narrative concentration over time"); plt.show()
    display(nf.describe())
else:
    print("No narrative features — run the embed stage first.")


## 4. Baselines + explainability
Naive (majority/persistence) + XGBoost on TimeSeriesSplit, then **SHAP** + **permutation
importance** on the tree to see *which feature families actually drive it* — news vs
cross-asset vs technicals. This is the honest "does the news signal earn its place?" check.

In [ ]:
from sentisense.models.baselines import run_baselines
res = run_baselines(mt)
display(pd.DataFrame(res).T)


In [ ]:
# Fit one XGBoost on a chronological train slice for explainability.
# (Trees need no scaling, and SHAP wants NAMED features — so we slice mt directly
#  rather than chronological_split, which returns scaled, unnamed arrays.)
import xgboost as xgb
y = mt["Target"].values.astype(int); X = mt.drop(columns=["Target"])
n = len(mt); ntr = int(n * 0.7); nva = int(n * 0.15)
Xtr, ytr = X.iloc[:ntr], y[:ntr]
Xte, yte = X.iloc[ntr + nva:], y[ntr + nva:]
pos = max(int(ytr.sum()), 1); neg = max(len(ytr) - int(ytr.sum()), 1)
clf = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.03,
                        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=neg / pos,
                        eval_metric="logloss", random_state=42, verbosity=0)
clf.fit(Xtr, ytr)

import shap
expl = shap.TreeExplainer(clf)
sv = expl.shap_values(Xte)
shap.summary_plot(sv, Xte, max_display=20, show=True)


In [ ]:
# Aggregate SHAP by feature FAMILY -> which signal source matters most.
absmean = np.abs(sv).mean(axis=0)
fam = {"news": 0.0, "cross_asset": 0.0, "technical": 0.0, "other": 0.0}
for col, v in zip(Xte.columns, absmean):
    fam[feat_family(col)] += v
fam = pd.Series(fam).sort_values()
fig, ax = plt.subplots(figsize=(7, 3)); fam.plot.barh(ax=ax, color="#6366f1")
ax.set_title("Total |SHAP| by feature family — where the signal comes from"); plt.show()

from sklearn.inspection import permutation_importance
pi = permutation_importance(clf, Xte, yte, n_repeats=10, random_state=42, scoring="roc_auc")
imp = pd.Series(pi.importances_mean, index=Xte.columns).sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(8, 5)); imp[::-1].plot.barh(ax=ax, color="#0ea5e9")
ax.set_title("Permutation importance (ROC-AUC drop), top 15"); plt.show()


## 5. LSTM tuning — Optuna visualizations
Loads the persisted studies (score-LSTM + embedding-LSTM) from the project DB. If none
exist yet, runs a SHORT demo search so the plots render. The full HPO is the long
headless `tune` stage.

In [ ]:
import os
import optuna
from sentisense.hpo.optuna_lstm import STUDY_SCORES, has_completed_trials
storage = get_connection_url()


def load_or_none(name):
    try:
        s = optuna.load_study(study_name=name, storage=storage)
        return s if has_completed_trials(s) else None
    except Exception:
        return None


study = load_or_none(STUDY_SCORES)
if study is None:
    print("No completed score-LSTM study found — running a SHORT demo (5 trials)…")
    from sentisense.hpo import run_hpo
    os.environ["SENTISENSE_OPTUNA_TRIALS"] = "5"
    study = run_hpo(ml, n_trials=5, study_name="demo_scores")
print("best value:", round(study.best_value, 4))
print("best params:", study.best_params)


In [ ]:
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances
plot_optimization_history(study); plt.title("Optuna optimization history (val ROC-AUC)"); plt.show()
plot_param_importances(study); plt.title("Hyperparameter importances"); plt.show()


## 6. Holdout evaluation + explainability
Sacred-test metrics for the score-LSTM (calibrated + threshold-tuned), ROC, a
**reliability (calibration) diagram**, and the confusion matrix.

In [ ]:
from sentisense.hpo import final_holdout_eval
summary_s, proba_s, labels_s = final_holdout_eval(ml, study.best_params, n_seeds=2)
display(pd.DataFrame(summary_s).T)


In [ ]:
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.calibration import calibration_curve

p, yv = proba_s.values, labels_s.values
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fpr, tpr, _ = roc_curve(yv, p); axes[0].plot(fpr, tpr, label=f"AUC={auc(fpr, tpr):.3f}")
axes[0].plot([0, 1], [0, 1], "--", c="navy"); axes[0].set_title("ROC (score-LSTM, holdout)"); axes[0].legend()
frac_pos, mean_pred = calibration_curve(yv, p, n_bins=10, strategy="quantile")
axes[1].plot(mean_pred, frac_pos, "o-"); axes[1].plot([0, 1], [0, 1], "--", c="gray")
axes[1].set_title("Reliability diagram (isotonic-calibrated)"); axes[1].set_xlabel("pred prob"); axes[1].set_ylabel("observed")
cm = confusion_matrix(yv, (p > 0.5).astype(int))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[2], xticklabels=["Down", "Up"], yticklabels=["Down", "Up"])
axes[2].set_title("Confusion matrix @0.5"); plt.tight_layout(); plt.show()


## 7. Backtest — equity curve, Sharpe, max drawdown
Long-when-predicted-up vs Buy&Hold on the sacred test window, using real TA-125 returns.
This is the "is it tradeable" view — directional accuracy != P&L.

In [ ]:
from sentisense.constants import TA125_CSV
ta = pd.read_csv(TA125_CSV)
ta["Date"] = pd.to_datetime(ta["Date"], errors="coerce")
ta = ta.dropna(subset=["Date"]).set_index("Date").sort_index()
price = ta["Price"].astype(str).str.replace(",", "", regex=False).astype(float)
ret = price.pct_change()

dates = proba_s.index
nxt_ret = ret.reindex(dates).shift(-1).fillna(0.0)   # next-day return realised on the prediction
signal = (proba_s.values > 0.5).astype(float)        # long if up predicted, else flat
strat = signal * nxt_ret.values
bh = nxt_ret.values


def sharpe(r):
    r = np.asarray(r); return float(np.sqrt(252) * r.mean() / (r.std() + 1e-9))


def maxdd(eq):
    peak = np.maximum.accumulate(eq); return float(((eq - peak) / peak).min())


eq_s = np.cumprod(1 + strat); eq_bh = np.cumprod(1 + bh)
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(dates, eq_s, label=f"LSTM long/flat (Sharpe {sharpe(strat):.2f}, maxDD {maxdd(eq_s):.1%})")
ax.plot(dates, eq_bh, label=f"Buy & Hold (Sharpe {sharpe(bh):.2f}, maxDD {maxdd(eq_bh):.1%})", c="gray")
ax.set_title("Holdout equity curve"); ax.legend(); plt.show()
print(f"Strategy total return: {eq_s[-1] - 1:+.1%} | Buy&Hold: {eq_bh[-1] - 1:+.1%}")


## 8. Verdict — how to read this

- **Beat the baseline?** Compare holdout accuracy/AUC (§6) to the majority-class + persistence
  baselines (§4) and the Buy&Hold Sharpe (§7). Directional accuracy near the up-rate with
  AUC≈0.5 = no real edge.
- **Where does signal come from?** §4 SHAP-by-family answers whether news earns its place
  vs cross-asset/technical features.
- **Is it calibrated?** §6 reliability diagram — if points hug the diagonal, the
  probabilities (and any abstention/trading thresholds built on them) are trustworthy.
- **Is it tradeable?** §7 Sharpe + max drawdown vs Buy&Hold. Higher directional accuracy
  with worse Sharpe means the model is right on small days and wrong on big ones.
- **Embedding LSTM + ensemble** live in the pipeline `final` stage; this notebook focuses
  on the score-LSTM track for explainability — swap `ml`→embedding dataset + `STUDY_EMB`
  to reproduce the embedding track here.
